In [1]:
from dd_cleaner.notebook_utils import (
    init_notebook_session, 
    get_raw_data, 
    get_cleaned_data, 
    get_tagged_entities,
    get_attributes_by_tag
)

# Initialize the session pointing to the parent directory (tests/)
coord, artifacts_df = init_notebook_session("..")

# Display the artifacts table
artifacts_df

✅ Notebook session initialized for workspace: /home/rajiv/programming/dd_parser_migration/sba_migration

Available Artifacts:

Artifact Name                           File Name  \
0                         Raw Data                   sba_loans_raw.csv   
1                     Cleaned Data             sba_loans_raw_clean.csv   
2             Tagged Entities (DD)  sba_loans_raw_analysis_results.csv   
3  Cleaning Recommendations Report         cleaning_recommendations.md   
4                 Profiling Report   sba_loans_raw_profiling_report.md   
5                   Handshake File         parser_cleaner_handshake.md   
6                  Quarantine File        sba_loans_raw_quarantine.csv   

                                            Location  Exists  
0                             data/sba_loans_raw.csv    True  
1            data/dd_cleaner/sba_loans_raw_clean.csv    True  
2  documents/dd_analysis_results/sba_loans_raw_an...    True  
3   documents/dd_cleaner/cleaning_recommendations.md    True  
4  documents/dd_cleaner/sba_loans_raw_profiling_r...    True  
5   documents/dd_cleaner/parser_cleaner_handshake.md    True  
6       data/quarantine/sba_loans_raw_quarantine.csv   False

,Artifact Name,File Name,Location,Exists
0,Raw Data,sba_loans_raw.csv,data/sba_loans_raw.csv,True
1,Cleaned Data,sba_loans_raw_clean.csv,data/dd_cleaner/sba_loans_raw_clean.csv,True
2,Tagged Entities (DD),sba_loans_raw_analysis_results.csv,documents/dd_analysis_results/sba_loans_raw_an...,True
3,Cleaning Recommendations Report,cleaning_recommendations.md,documents/dd_cleaner/cleaning_recommendations.md,True
4,Profiling Report,sba_loans_raw_profiling_report.md,documents/dd_cleaner/sba_loans_raw_profiling_r...,True
5,Handshake File,parser_cleaner_handshake.md,documents/dd_cleaner/parser_cleaner_handshake.md,True
6,Quarantine File,sba_loans_raw_quarantine.csv,data/quarantine/sba_loans_raw_quarantine.csv,False


In [2]:
# Load the raw data
df_raw = get_raw_data(coord)

# Load the cleaned data produced by the pipeline
df_clean = get_cleaned_data(coord)

# Load the metadata/tagged entities
df_tags = get_tagged_entities(coord)

In [3]:
df_clean.columns

Index(['asofdate', 'program', 'locationid', 'borrname', 'borrstreet',
       'borrcity', 'borrstate', 'borrzip', 'grossapproval', 'approvaldate',
       'approvalfy', 'firstdisbursementdate', 'processingmethod', 'subprogram',
       'terminmonths', 'naicscode', 'naicsdescription', 'franchisecode',
       'franchisename', 'projectcounty', 'projectstate', 'sbadistrictoffice',
       'congressionaldistrict', 'businesstype', 'businessage', 'loanstatus',
       'paidinfulldate', 'chargeoffdate', 'grosschargeoffamount',
       'jobssupported', 'collateralind'],
      dtype='str')

In [4]:
# Select rows where 'firstdisbursementdate' is null
null_disbursement_rows = df_clean[df_clean['firstdisbursementdate'].isna()]
null_disbursement_rows.loanstatus.value_counts()

loanstatus
CANCELED        14302
NOT FUNDED       8261
PAID IN FULL       95
Name: count, dtype: int64

In [5]:
df_clean.loanstatus.value_counts()

loanstatus
CURRENT               54435
PREPAID IN FULL       34428
CANCELED              14302
NOT FUNDED             8261
IN CATCH-UP            1061
CHARGED-OFF             910
PAID IN FULL            717
PAID IN FULL (LIQ)      669
PURCH(NOT C/O)          597
2 MONTHS PAST DUE       267
1 MONTH PAST DUE        150
IN DEFERMENT            100
3 MONTHS PAST DUE        79
4 MONTHS PAST DUE        43
6 MONTHS PAST DUE        16
5 MONTHS PAST DUE        14
7 MONTHS PAST DUE        12
>9 MONTHS PAST DUE       12
8 MONTHS PAST DUE        12
9 MONTHS PAST DUE         7
ASSET SALE                2
Name: count, dtype: int64

## Explanation of Status from Gemini

Based on the dataset structure you shared, you are looking at an export of SBA loan portfolio performance data (likely 7(a) or 504 loan data) tracking the status of 115,134 individual accounts.
In this context, the statuses track whether the loan is active, successfully closed out, or in a state of distress/default.
------------------------------
## 🟢 Active & Good Standing Statuses
These loans are operational and behaving exactly as intended.

* CURRENT ($54,435$ loans): The borrower is paying on time; the account is completely up to date.
* IN DEFERMENT ($100$ loans): The SBA or the lender has granted a temporary, authorized pause on monthly payments (often used during economic hardships or disasters). [1, 2] 

## 🔵 Successfully Closed Statuses
These loans have reached the end of their lifecycle with no losses incurred.

* PREPAID IN FULL ($34,428$ loans): The borrower paid off the entire loan balance early, ahead of the original maturity date.
* PAID IN FULL ($717$ loans): The borrower successfully completed their regular payment schedule and paid off the loan naturally. [1, 3] 

## 🟡 Delinquent Statuses (Past Due)
These loans are actively falling behind on their payments, broken down by exactly how many monthly cycles have been missed.

* 1 MONTH PAST DUE ($150$ loans) to 9 MONTHS PAST DUE ($7$ loans): The borrower has missed sequential monthly payments.
* >9 MONTHS PAST DUE ($12$ loans): Severe long-term delinquency where the borrower has missed 10 or more consecutive payments but the loan hasn't been officially written off yet. [1] 

## 🔴 Distress, Liquidation, & Default Statuses
These statuses indicate that a business has failed to pay, forcing the lender and the SBA to step in to recover funds.

* IN CATCH-UP ($1,061$ loans): The borrower fell behind but has entered into an approved agreement or payment plan to pay off the past-due balance. [1] 
* PURCH(NOT C/O) [Purchased, Not Charged Off] ($597$ loans): The borrower defaulted, so the SBA "purchased" the guaranteed portion of the loan from the commercial bank. However, it is not yet charged off because active collection efforts or asset liquidations are still ongoing. [4, 5] 
* PAID IN FULL (LIQ) [Liquidation] ($669$ loans): The loan was fully paid off, but only because the lender seized and sold off the business's collateral (real estate, equipment, inventory) to cover the debt. [4] 
* CHARGED-OFF ($910$ loans): The borrower defaulted, all collateral has been liquidated, and the SBA has officially written the remaining balance off as an uncollectible loss. [4] 
* ASSET SALE ($2$ loans): The non-performing loan note itself was sold at a discount to an outside debt buyer/investor to get it off the books. [5] 

## ⚪ Non-Origination Statuses
These applications never actually became live loans.

* CANCELED ($14,302$ loans): The loan approval was withdrawn or voided before closing docs were finalized (either by the borrower or the lender).
* NOT FUNDED ($8,261$ loans): The loan process was finalized and approved, but the capital was never successfully disbursed or accepted by the borrower. [3, 6] 

------------------------------
If you are building a predictive risk model or analyzing portfolio health with this data, I can help you write the Python code to group these into clean categories (e.g., Default vs. Non-Default). Would you like to see how to aggregate these for a credit risk analysis?

[1] [https://www.mintos.com](https://www.mintos.com/blog/everything-you-wanted-to-know-about-loan-lifecycle-and-status/)
[2] [https://landlordinvest.com](https://landlordinvest.com/frequently-asked-questions/what-are-the-different-loan-statuses-and-what-do-they-mean)
[3] [https://www.saarathi.ai](https://www.saarathi.ai/blog/loan-stuck-in-%E2%80%98under-review%E2%80%99-what-that-status-actually-means)
[4] [https://www.linkedin.com](https://www.linkedin.com/pulse/understanding-non-performing-sba-loan-lance-sexton-ghucc)
[5] [https://ebitcommunity.com](https://ebitcommunity.com/p/what-happens-if-you-default-on-an-sba-7-a-acquisition-loan)
[6] [https://www.kotak.bank.in](https://www.kotak.bank.in/en/stories-in-focus/loans/personal-loan/what-is-loan-disbursement.html)


In [1]:

pna = df_clean.firstdisbursementdate > df_clean.paidinfulldate


NameError: name 'df_clean' is not defined

In [7]:
df_clean[pna]

,asofdate,program,locationid,borrname,borrstreet,borrcity,borrstate,borrzip,grossapproval,approvaldate,...,sbadistrictoffice,congressionaldistrict,businesstype,businessage,loanstatus,paidinfulldate,chargeoffdate,grosschargeoffamount,jobssupported,collateralind
6,3/31/2026,504,188319,"Shield Security Systems, L.L.C.",7456 West 5th Avenue.,Lakewood,CO,80226,757000,10/2/2009,...,COLORADO DISTRICT OFFICE,1.0,CORPORATION,Less than 4 years old but at least 3,PREPAID IN FULL,5/31/2016,NaN,0.0,40,True
12,3/31/2026,504,188280,KT&apos;s Bowling Lanes,1501 S. Washington Avenue.,Emmett,ID,83617,111000,10/2/2009,...,BOISE DISTRICT OFFICE,1.0,CORPORATION,Unanswered,PREPAID IN FULL,10/31/2017,NaN,0.0,10,True
13,3/31/2026,504,188239,Cervesia Gratis Inc,1483 Duane St.,Astoria,OR,97103,273000,10/2/2009,...,PORTLAND DISTRICT OFFICE,1.0,CORPORATION,Less than 4 years old but at least 3,PREPAID IN FULL,11/30/2014,NaN,0.0,8,True
14,3/31/2026,504,188328,K & M Transmissions LLC,10135 Earl Dr.,St. john,IN,46373,340000,10/5/2009,...,INDIANA DISTRICT OFFICE,1.0,CORPORATION,Less than 4 years old but at least 3,PREPAID IN FULL,2/29/2016,NaN,0.0,3,True
17,3/31/2026,504,188169,TST Tooling Software LLC,6547 Dixie Hwy.,Clarkston,MI,48346,184000,10/5/2009,...,MICHIGAN DISTRICT OFFICE,8.0,CORPORATION,Less than 4 years old but at least 3,PREPAID IN FULL,3/31/2019,NaN,0.0,10,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116295,3/31/2026,504,188309,ACS HOLDING LLC,5577 Brooks Street,Montclair,CA,91763,1082000,6/19/2020,...,SANTA ANA DISTRICT OFFICE,27.0,CORPORATION,Existing or more than 2 years old,PREPAID IN FULL,7/31/2024,NaN,0.0,7,True
116311,3/31/2026,504,188216,Brigham Larson Holdings LLC,1497 South State Street,OREM,UT,84097,546000,6/19/2020,...,UTAH DISTRICT OFFICE,3.0,CORPORATION,Existing or more than 2 years old,PREPAID IN FULL,8/31/2024,NaN,0.0,2,True
116315,3/31/2026,504,188309,"The Broken Token, LLC",2440 Grand Avenue,VISTA,CA,92081,1071000,6/22/2020,...,SAN DIEGO DISTRICT OFFICE,49.0,CORPORATION,Existing or more than 2 years old,PREPAID IN FULL,10/31/2023,NaN,0.0,43,True
116327,3/31/2026,504,440699,Horse Thief Campground & RV Resort,24391 SD Highway 87,Hill City,SD,57745,650000,6/22/2020,...,SOUTH DAKOTA DISTRICT OFFICE,0.0,CORPORATION,Existing or more than 2 years old,PREPAID IN FULL,12/31/2024,NaN,0.0,4,True


In [8]:
import os
import sys
import pandas as pd

# 1. Ensure the project root is in the path
project_root = "/home/rajiv/programming/dd_parser_migration/sba_migration"
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Now import your custom logic
try:
    from scripts.domain_logic import filter_null_disbursements
    print("✅ Successfully imported filter_null_disbursements")
    
    # 3. Test the filter logic
    # Loading a few rows for verification
    valid_index = filter_null_disbursements(df_clean)

    print(f"Total Rows Sampled: {len(df_clean)}")
    print(f"Rows after filtering Null Disbursements: {len(valid_index)}")
    print(f"Nulls detected in sample: {df_clean['firstdisbursementdate'].isna().sum()}")

except ModuleNotFoundError as e:
    print(f"❌ Still unable to find the module: {e}")
    print(f"Current Working Directory: {os.getcwd()}")
    print(f"System Path: {sys.path}")


✅ Successfully imported filter_null_disbursements
Total Rows Sampled: 116345
Rows after filtering Null Disbursements: 93436
Nulls detected in sample: 22909


In [10]:
import pandas as pd
import numpy as np
from scripts.domain_logic import invalid_first_disbursement_date

# Load data
project_root = "/home/rajiv/programming/dd_parser_migration/sba_migration"


# Ensure dates are treated correctly for the test
df_clean['firstdisbursementdate'] = pd.to_datetime(df_clean['firstdisbursementdate'], errors='coerce')
df_clean['paidinfulldate'] = pd.to_datetime(df_clean['paidinfulldate'], errors='coerce')

# Run filter
keep_idx = invalid_first_disbursement_date(df_clean)
removed_count = len(df_clean) - len(keep_idx)

print(f"Total records: {len(df_clean)}")
print(f"Records to keep: {len(keep_idx)}")
print(f"Records filtered out (Disbursed > Paid): {removed_count}")

# Show a sample of what was removed (if any)
if removed_count > 0:
    print("\nSample of removed records:")
    print(df_clean.loc[~df_clean.index.isin(keep_idx), ['firstdisbursementdate', 'paidinfulldate']].head())


Total records: 116345
Records to keep: 116327
Records filtered out (Disbursed > Paid): 18

Sample of removed records:
      firstdisbursementdate paidinfulldate
44203            2023-04-12     2023-03-31
51224            2026-04-15     2026-02-28
68354            2026-04-15     2026-02-28
69194            2026-04-15     2026-02-28
69756            2026-04-15     2026-02-28
